# DATA WRANGLING

## SETUP AND IMPORT

In [ ]:
import duckdb
import pandas as pd
import sqlite3
from pathlib import Path
import holidays
import os
import numpy as np
from functions import filter_train_type, create_features

os.chdir("/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project")

con = duckdb.connect("data/train.duckdb")

con.execute("""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('data/train_delay_with_weather.parquet')
            """)

df_raw = con.execute("SELECT * FROM train_delay").fetchdf()

## DATA INSPECTION

In [9]:
print(df_raw.info())                      
# print(df_raw.head(5))                      
# print(df_raw["delay_in_min"].describe())  
# print(df_raw["train_type"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2175996 entries, 0 to 2175995
Data columns (total 15 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   canceled           bool          
 6   arrival_planned    datetime64[us]
 7   arrival_real       datetime64[us]
 8   departure_planned  datetime64[us]
 9   departure_real     datetime64[us]
 10  delay              int32         
 11  temperature        float64       
 12  precipitation      float64       
 13  wind_speed         float64       
 14  weather_status     object        
dtypes: bool(1), datetime64[us](4), float64(3), int32(1), int64(1), object(5)
memory usage: 226.2+ MB
None


## DATA CLEANING

In [ ]:
### FILTER TRAIN TYPES (ONLY ICE AND IC)
df_cleaned = filter_train_type(df_raw)

### REMOVE RIDES THAT WERE CANCELED
df_cleaned = (
    df_cleaned.groupby("ride_id")
      .filter(lambda g: not g["canceled"].any())
)

### REMOVE UNECESSARY COLUMNS
df_cleaned = df_cleaned.drop(columns=["weather_status", "canceled"])

### RENAME COLUMNS
df_cleaned = df_cleaned.rename(columns={
    "station_name": "station_current",
    "final_destination_station": "station_dest"})

# REARRANGE COLUMNS
df_cleaned = df_cleaned.loc[:, ["ride_id", "train_name", "train_type", "station_current", "station_dest",
               "departure_planned", "departure_real", "arrival_planned", "arrival_real",
                "temperature", "precipitation", "wind_speed", "delay"]]


print(df_cleaned.columns.tolist())
print(df_cleaned.info())    


['ride_id', 'train_name', 'train_type', 'station_current', 'station_dest', 'departure_planned', 'departure_real', 'arrival_planned', 'arrival_real', 'temperature', 'precipitation', 'wind_speed', 'delay']
<class 'pandas.core.frame.DataFrame'>
Index: 1842266 entries, 0 to 2175995
Data columns (total 13 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   departure_planned  datetime64[us]
 6   departure_real     datetime64[us]
 7   arrival_planned    datetime64[us]
 8   arrival_real       datetime64[us]
 9   temperature        float64       
 10  precipitation      float64       
 11  wind_speed         float64       
 12  delay              int32         
dtypes: datetime64[us](4), float64(3), int32(1), int64(1), object(4)
memory usage: 189.7+ MB
None


## NA ANALYSIS

In [11]:
# number of na per column
print(df_cleaned.isnull().sum(axis = 0))

# missings in time_current_departure_planned and time_current_arrival_plannned
# are due to first or last station that do not have either arrival time (start station) or departure time (destination station)
len(df_cleaned["ride_id"].unique())

ride_id                   0
train_name                0
train_type                0
station_current           0
station_dest              0
departure_planned    200154
departure_real       200031
arrival_planned      200154
arrival_real         200036
temperature               0
precipitation             0
wind_speed                0
delay                     0
dtype: int64


200154

## CREATE FEATURES

### STATION START

In [ ]:
### REASON: ALSO EXISTS IN API DATA
### AFTER THAT, WE CAN APPLY FUNCTIONS TO BOTH DATASETS

# sorting
df_cleaned.sort_values(by=["ride_id", "departure_planned"])

# grouping
grouped = df_cleaned.groupby("ride_id")

# create station_start
df_cleaned["station_start"] = grouped["station_current"].transform("first")

### OTHER FEATURES (WITH FUNCTION)

In [ ]:
df_features = create_features(df_cleaned)

### CHECK CORRECTNESS OF DATASET

In [25]:
df_features.describe()

,ride_id,departure_planned,departure_real,arrival_planned,arrival_real,temperature,precipitation,wind_speed,delay,stops_total,departure_planned_start,arrival_planned_dest,travel_time,weekday,month,hour,dwell_time_planned,stop_index,time_since_start_planned,share_ride_time
count,1.842266e+06,1642112,1642235,1642112,1642230,1.842266e+06,1.842266e+06,1.842266e+06,1.842266e+06,1.842266e+06,1842266,1842266,1.842266e+06,1.842266e+06,1.842266e+06,1.842266e+06,1.842266e+06,1.842266e+06,1.842266e+06,1.842266e+06
mean,1.190158e+05,2025-04-20 00:33:01.594458,2025-04-20 00:38:15.188350,2025-04-20 01:05:16.456526,2025-04-20 01:07:21.069289,1.074211e+01,2.001426e+00,1.603990e+01,9.585715e+00,1.084977e+01,2025-04-19 05:58:48.648185,2025-04-19 11:27:11.020787,3.283729e+02,2.953455e+00,7.781153e+00,1.345168e+01,2.369372e+00,5.924884e+00,1.619217e+02,4.928262e-01
min,8.300000e+01,2024-07-01 00:01:00,2024-07-01 00:02:00,2024-07-01 00:07:00,2024-07-01 00:09:00,-7.850000e+00,0.000000e+00,2.200000e+00,-5.300000e+01,2.000000e+00,2024-07-01 00:01:00,2024-07-01 02:50:00,5.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,6.556700e+04,2024-11-30 13:58:45,2024-11-30 13:57:30,2024-11-30 14:31:00,2024-11-30 14:29:00,4.890000e+00,0.000000e+00,1.150000e+01,1.000000e+00,8.000000e+00,2024-11-30 06:13:00,2024-11-30 11:42:00,2.660000e+02,1.000000e+00,5.000000e+00,9.000000e+00,1.000000e+00,3.000000e+00,4.300000e+01,1.538462e-01
50%,1.207845e+05,2025-04-20 08:45:30,2025-04-20 08:40:00,2025-04-20 09:24:00,2025-04-20 09:16:00,1.062000e+01,2.000000e-01,1.530000e+01,3.000000e+00,1.100000e+01,2025-04-19 07:46:00,2025-04-19 14:00:00,3.170000e+02,3.000000e+00,8.000000e+00,1.400000e+01,2.000000e+00,5.000000e+00,1.420000e+02,4.864865e-01
75%,1.724380e+05,2025-09-18 17:43:15,2025-09-18 17:55:00,2025-09-18 18:23:00,2025-09-18 18:28:00,1.667000e+01,2.100000e+00,1.960000e+01,1.200000e+01,1.300000e+01,2025-09-17 08:38:00,2025-09-17 14:45:00,3.850000e+02,5.000000e+00,1.100000e+01,1.800000e+01,3.000000e+00,8.000000e+00,2.550000e+02,8.285714e-01
max,2.332690e+05,2025-12-31 23:50:00,2025-12-31 23:50:00,2025-12-31 23:58:00,2025-12-31 23:54:00,3.063000e+01,5.750000e+01,5.900000e+01,4.060000e+02,2.900000e+01,2025-12-31 21:49:00,2025-12-31 23:58:00,8.390000e+02,6.000000e+00,1.200000e+01,2.300000e+01,6.900000e+01,2.900000e+01,8.390000e+02,1.000000e+00
std,6.540309e+04,NaN,NaN,NaN,NaN,7.173175e+00,3.994898e+00,5.952543e+00,1.640014e+01,4.004719e+00,NaN,NaN,1.176464e+02,1.993495e+00,3.401780e+00,5.444643e+00,2.685650e+00,3.882317e+00,1.325944e+02,3.514205e-01


### HISTORICAL AGGREGATION --> MOVE TO PIPELINE!

In [15]:
# function to add expanding features
def add_expanding_features(df, group_cols, prefix):
    df = df.sort_values(group_cols + ["time_current_departure_real"]) # sort by real time because otherwise expanding stats might include info from future

    grouped = df.groupby(group_cols, sort=False)["delay"]

    df[f"{prefix}_avg"] = grouped.transform(lambda x: x.shift().expanding().mean())
    df[f"{prefix}_std"] = grouped.transform(lambda x: x.shift().expanding().std())
    df[f"{prefix}_q90"] = grouped.transform(lambda x: x.shift().expanding().quantile(0.9))
    df[f"{prefix}_count"] = grouped.transform(lambda x: x.shift().expanding().count()
)

    return df

# derive time features from real time
features_station["hour"] = features_station["time_current_departure_real"].dt.hour
features_station["weekday"] = features_station["time_current_departure_real"].dt.dayofweek

# apply expanding features
features_station = add_expanding_features(features_station, ["station_current"], "hist_delay_station")   
features_station = add_expanding_features(features_station, ["station_current", "hour"], "hist_delay_station_hour")  
features_station = add_expanding_features(features_station, ["station_current", "weekday"], "hist_delay_station_day")  

/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby(group_cols, sort=False)["delay"]
/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby(group_cols, sort=False)["delay"]
/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or ob

## TARGET TRANSFORMATION

In [26]:
df_features["delay"] = pd.cut(
    df_features["delay"],
    bins=[-np.inf, 60, 120, np.inf],
    labels=[0, 1, 2],
)

df_features["delay"].value_counts(normalize=True)


delay
0    0.980727
1    0.017127
2    0.002146
Name: proportion, dtype: float64